In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "4,5,6,7"

from transformers import AutoModelForCausalLM, AutoTokenizer
import numpy as np
import torch
from tqdm import tqdm

import sys
import string

sys.path.append('../utils/')
import config

sys.path.append('../data/')

sys.path.append('../')
# model = AutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-v0.1", low_cpu_mem_usage=True, torch_dtype=torch.float16,
#                                                  trust_remote_code=True).cuda()
# tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1")

model =  AutoModelForCausalLM.from_pretrained("/home/ubuntu/DPO/huggingface_trl_dpo/dpo_ultrafeedback_binarized_cshin/checkpoint-15000", 
                                              low_cpu_mem_usage=True, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained("/home/ubuntu/DPO/huggingface_trl_dpo/dpo_ultrafeedback_binarized_cshin/checkpoint-15000")

# model = AutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1", torch_dtype=torch.float16).cuda()
# tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1", use_fast=False)

In [ ]:
device = "cuda"  
# model.to(device)
n_layers = model.config.num_hidden_layers
n_heads = model.config.num_attention_heads

In [ ]:
from baukit import TraceDict

def get_single_activation(model, query):
    MLPS_OUT = [f"model.layers.{i}.mlp.gate_proj" for i in range(model.config.num_hidden_layers)]
    input_ids = tokenizer(query, return_tensors="pt").input_ids.cuda()
    with torch.no_grad():
        with TraceDict(model, MLPS_OUT) as ret:
            output = model(input_ids, output_hidden_states = True)
        mlp_out = [ret[mlp_].output.squeeze().detach().cpu() for mlp_ in MLPS_OUT]
        mlp_out = torch.stack(mlp_out, dim = 0).squeeze().numpy()
    return mlp_out[:, -1, :]

In [ ]:
def get_insights_pair_emb(model, pos_insights, neg_insights):
    p_embed = []
    n_embed = []
    for f_pos, f_neg in tqdm(zip(pos_insights, neg_insights)):
        try:
            p_embed_ = get_single_activation(model, f_pos)
            n_embed_ = get_single_activation(model, f_neg)
            p_embed.append(p_embed_)
            n_embed.append(n_embed_)
        except Exception as e:
            raise e
    return p_embed, n_embed

In [ ]:
layers_to_edit = [i for i in range(0, n_layers)]

In [ ]:
from utils.data_utils import set_seed
from transformers import StoppingCriteriaList, StoppingCriteria
import json

sys.path.append('../data/')
from functools import partial
from utils.inference import vanila_inference, StopOnTokens
import pandas as pd

SEED = 0
set_seed(SEED)

max_new_tokens = 1024
n_samples = 500

outdir=f'results/tqa_100p'

import pandas as pd
df = pd.read_csv('self_generated_data/tqa_open_generation.csv')

from datasets import load_dataset
dataset = load_dataset("truthful_qa", "generation")

from utils.data_utils import load_truthfulqa_template
template_path = '../data/truthful-qa' # '../data/hh-rlhf'
template = load_truthfulqa_template(template_path)
fschat = template['fschat']
print(fschat)

In [ ]:
outdir

In [ ]:
from scipy import linalg 

def get_layer_wise_proj(pos_emb, neg_emb):
    proj = {i:[] for i in range(n_layers)}
    for i in tqdm(range(n_layers)):
        matrix = []
        for p in range(len(pos_emb)):
            p_emb = pos_emb[p][i,:]
            n_emb = neg_emb[p][i,:]
            diff_ = n_emb-p_emb
            if np.linalg.norm(diff_) == 0:
                continue
            else:
                matrix.append(diff_)
        matrix = np.vstack(matrix)
        u,s,v = linalg.svd(matrix, full_matrices=False)
        proj[i] = v[0,:]
    return proj

In [ ]:
def get_class_means(pos_emb, neg_emb):
    proj = {i:[] for i in range(n_layers)}
    for i in tqdm(range(n_layers)):
        p_emb_all = []
        n_emb_all = []
        for p in range(len(pos_emb)):
            p_emb = pos_emb[p][i,:]
            p_emb_all.append(p_emb)
        for p in range(len(neg_emb)):
            n_emb = neg_emb[p][i,:]
            n_emb_all.append(n_emb)
        
        p_emb_all = np.vstack(p_emb_all)
        _,_,v_pos = linalg.svd(p_emb_all, full_matrices=False)
        n_emb_all = np.vstack(n_emb_all)
        _,_,v_neg = linalg.svd(n_emb_all, full_matrices=False)
        # proj[i] = (v_pos[0,:], v_neg[0,:])
        proj[i] = (np.mean(p_emb_all,axis=0), np.mean(n_emb_all,axis=0), v_pos, v_neg)
    return proj

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

def get_probes(pos_emb, neg_emb):
    proj = {i:{'clf': [], 'loss': None} for i in range(n_layers)}
    for i in tqdm(range(n_layers)):
        p_emb_all = []
        n_emb_all = []
        for p in range(len(pos_emb)):
            p_emb = pos_emb[p][i,:]
            p_emb_all.append(p_emb)
        for p in range(len(neg_emb)):
            n_emb = neg_emb[p][i,:]
            n_emb_all.append(n_emb)
        p_emb_all = np.vstack(p_emb_all)
        n_emb_all = np.vstack(n_emb_all)
        X = np.vstack((p_emb_all, n_emb_all))
        y = np.hstack(([1 for i in range(len(p_emb_all))],[0 for i in range(len(n_emb_all))]))
        # X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        ccs = CCS(n_emb_all, p_emb_all)
        best_loss = ccs.repeated_train()
        
        # Evaluate
        ccs_acc = ccs.get_acc(n_emb_all, p_emb_all, y)
        print("layer {} CCS accuracy: {}".format(i, ccs_acc))
        proj[i]['clf'] = ccs.best_probe
        proj[i]['loss'] = best_loss
    return proj

In [ ]:
import copy
import torch.nn as nn

class CCS(object):
    def __init__(self, x0, x1, nepochs=1000, ntries=10, lr=1e-3, batch_size=-1, 
                 verbose=False, device="cuda", linear=True, weight_decay=0.01, var_normalize=False):
        # data
        self.var_normalize = var_normalize
        self.x0 = x0
        self.x1 = x1
        self.d = self.x0.shape[-1]

        # training
        self.nepochs = nepochs
        self.ntries = ntries
        self.lr = lr
        self.verbose = verbose
        self.device = device
        self.batch_size = batch_size
        self.weight_decay = weight_decay
        
        # probe
        self.linear = linear
        self.initialize_probe()
        self.best_probe = copy.deepcopy(self.probe)

        
    def initialize_probe(self):
        self.probe = nn.Sequential(nn.Linear(self.d, 1))
        self.probe.to(self.device)    

    def normalize(self, x):
        """
        Mean-normalizes the data x (of shape (n, d))
        If self.var_normalize, also divides by the standard deviation
        """
        normalized_x = x - x.mean(axis=0, keepdims=True)
        if self.var_normalize:
            normalized_x /= normalized_x.std(axis=0, keepdims=True)

        return normalized_x

        
    def get_tensor_data(self):
        """
        Returns x0, x1 as appropriate tensors (rather than np arrays)
        """
        x0 = torch.tensor(self.x0, dtype=torch.float, requires_grad=False, device=self.device)
        x1 = torch.tensor(self.x1, dtype=torch.float, requires_grad=False, device=self.device)
        return x0, x1
    

    def get_loss(self, p0, p1):
        """
        Returns the CCS loss for two probabilities each of shape (n,1) or (n,)
        """
        informative_loss = (torch.min(p0, p1)**2).mean(0)
        consistent_loss = ((p0 - (1-p1))**2).mean(0)
        return informative_loss + consistent_loss


    def get_acc(self, x0_test, x1_test, y_test):
        """
        Computes accuracy for the current parameters on the given test inputs
        """
        x0 = torch.tensor(x0_test, dtype=torch.float, requires_grad=False, device=self.device)
        x1 = torch.tensor(x1_test, dtype=torch.float, requires_grad=False, device=self.device)
        with torch.no_grad():
            p0, p1 = self.best_probe(x0), self.best_probe(x1)
            p0 = torch.sigmoid(p0)
            p1 = torch.sigmoid(p1)
        p0_preds = (p0>0.5).float().detach().cpu().numpy().tolist()
        p1_preds = (p1>0.5).float().detach().cpu().numpy().tolist()
        predictions = p0_preds
        predictions.extend(p1_preds)
        acc = (predictions == y_test).mean()
        acc = max(acc, 1 - acc)
        return acc
    
        
    def train(self):
        """
        Does a single training run of nepochs epochs
        """
        x0, x1 = self.get_tensor_data()
        if len(x0) < len(x1):
            permutation = torch.randperm(len(x0))
        else:
            permutation = torch.randperm(len(x1))
        x0, x1 = x0[permutation], x1[permutation]
        
        # set up optimizer
        optimizer = torch.optim.AdamW(self.probe.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        
        batch_size = len(x0) if self.batch_size == -1 else self.batch_size
        nbatches = len(x0) // batch_size

        # Start training (full batch)
        for epoch in range(self.nepochs):
            ep_loss = 0
            for j in range(nbatches):
                x0_batch = x0[j*batch_size:(j+1)*batch_size]
                x1_batch = x1[j*batch_size:(j+1)*batch_size]
            
                # probe
                p0, p1 = self.probe(x0_batch), self.probe(x1_batch)
                p0 = torch.sigmoid(p0)
                p1 = torch.sigmoid(p1)
                # get the corresponding loss
                loss = self.get_loss(p0, p1)
                ep_loss += loss
                # update the parameters
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            ep_loss = ep_loss / nbatches
            if (epoch+1)/250 == 0:
                print(f"epoch {epoch} loss: {ep_loss}")

        return loss.detach().cpu().item()
    
    def repeated_train(self):
        best_loss = np.inf
        for train_num in range(self.ntries):
            self.initialize_probe()
            loss = self.train()
            if loss < best_loss:
                self.best_probe = copy.deepcopy(self.probe)
                best_loss = loss

        return best_loss

In [ ]:
def get_cosine_sim(v1, v2):
    return torch.dot(v1, v2)/(torch.linalg.vector_norm(v1)*torch.linalg.vector_norm(v2))
    
def get_interventions_dict(probes, class_means):
    interventions = {}
    best_loss_all = []
    for layer_idx in probes:
        best_loss_all.append(probes[layer_idx]['loss'])
    best_loss_all = np.array(best_loss_all)
    best_loss_sorted_idx = np.argsort(best_loss_all)

    running_mean = []
    for i in range(1, len(best_loss_sorted_idx)):
        losses = best_loss_all[best_loss_sorted_idx[:i]]
        running_mean.append(np.mean(losses))
    diffs = np.diff(np.array(running_mean))
    stop_edit_idx = np.argmax(diffs).flatten()[0]
    # if stop_edit_idx > 20:
    #     stop_edit_idx = 20
    layers_to_edit = best_loss_sorted_idx[:stop_edit_idx]
    for l_idx in tqdm(layers_to_edit):
        probe = probes[l_idx]['clf']
        mean_pos, mean_neg, v_pos, v_neg = class_means[l_idx]
        mean_pos = torch.Tensor(mean_pos).cuda()
        mean_neg = torch.Tensor(mean_neg).cuda()
        
        pos = torch.sigmoid(probe(mean_pos)).detach().cpu().numpy()[0]
        neg = torch.sigmoid(probe(mean_neg)).detach().cpu().numpy()[0]
        print(l_idx, pos, neg)
        if pos > .5 and neg < .5:
            harm_subspace=False
        elif neg > .5 and pos < .5:
            harm_subspace=True
        else: 
            interventions[f"model.layers.{l_idx}.mlp.gate_proj"] = ([], None, v_pos[0,:])
            continue
        
        for layer in probe.modules():
           if isinstance(layer, nn.Linear):
               coef = layer.weight.detach().cpu().numpy()
        # if len(probe) > 0:
        interventions[f"model.layers.{l_idx}.mlp.gate_proj"] = (coef.flatten(), harm_subspace, v_pos[0,:])
    return interventions

In [ ]:
def lt_modulated_proj(layer_output, layer_name, interventions):
    probe, is_harm, helpful_space = interventions[layer_name]
    layer_output = layer_output.squeeze() 
    if len(layer_output.shape) > 1:
        x_test = layer_output[-1,:]
    else:
        x_test = layer_output
    
    x_test = x_test.squeeze()
    probe = torch.Tensor(probe).to(torch.float16).to(model.device).squeeze() 
    helpful_space = torch.Tensor(helpful_space).to(torch.float16).to(model.device).squeeze() 

    if is_harm != None and is_harm==True:
        if probe.device != x_test.device:
            probe = probe.to(x_test.device)
        prod = torch.dot(x_test, probe)
        proj_harm = torch.dot(x_test, probe)/torch.linalg.vector_norm(probe)
        proj_harm = proj_harm * probe
        proj_harm = x_test - proj_harm

        if helpful_space.device != proj_harm.device:
            helpful_space = helpful_space.to(proj_harm.device)
        proj_help = torch.dot(proj_harm, helpful_space)/torch.linalg.vector_norm(helpful_space)
        proj_help = proj_harm * helpful_space
        proj_help = proj_harm + proj_help
        proj = proj_help 
    else:
        if helpful_space.device != x_test.device:
            helpful_space = helpful_space.to(x_test.device)
        proj_help = torch.dot(x_test, helpful_space)/torch.linalg.vector_norm(helpful_space)
        proj_help = x_test * helpful_space
        proj_help = x_test + proj_help
        proj = proj_help 
    
    if len(layer_output.shape) > 1:
        layer_output[-1,:] = proj
    else:
        layer_output = proj
        
    layer_output = layer_output.unsqueeze(0)
    layer_output = layer_output.to(torch.float16)
    layer_output = layer_output.to(model.device)
    return layer_output

In [ ]:
def convert_obj(item, instruction = None):
    q = item['question']
    if not instruction:
        tmp_ = f"Human: {q}\nAssistant: "
    else:
        tmp_ = f"Human: {q}\n{instruction}\nAssistant: "
    return tmp_

def convert_to_quesion_str(q, instruction = None):
    if not instruction:
        tmp_ = f"Human: {q}\nAssistant: "
    else:
        tmp_ = f"Human: {q}\n{instruction}\nAssistant: "
    return tmp_

In [ ]:
def get_insights(df_insights):
    questions = df_insights['question'].tolist()
    
    pos_rows = df_insights['truthful_ans']
    neg_rows = df_insights['malicious_ans']

    pos_samples = pos_rows.tolist()
    neg_samples = neg_rows.tolist()

    idxs_to_use = np.argwhere(np.array(pos_samples)!=np.array(neg_samples)).flatten()
    # print(len(idxs_to_use))

    questions_str_all = [convert_to_quesion_str(questions[i]) for i in idxs_to_use]
    pos_samples_con=[]
    neg_samples_cont=[]
    for i, (q,p) in enumerate(zip(questions_str_all, pos_samples)):
        try:
            pos_samples_con.append(q+p)
        except:
            continue
        
    for i, (q,n) in enumerate(zip(questions_str_all, neg_samples)):
        try:
            neg_samples_cont.append(q+n)
        except:
            continue
    return pos_samples_con, neg_samples_cont

In [ ]:
pos_samples, neg_samples = get_insights(df)

In [ ]:
pos_samples[0]

In [ ]:
neg_samples[0]

In [ ]:
print(len(pos_samples), len(neg_samples))

In [ ]:
random_idxs = np.random.choice(np.arange(len(pos_samples)),n_samples)
pos_samples = np.array(pos_samples)[random_idxs]
neg_samples = np.array(neg_samples)[random_idxs]

In [ ]:
pos_emb, neg_emb = get_insights_pair_emb(model, pos_samples, neg_samples)

In [ ]:
# layerwise_proj_neg = get_layer_wise_proj(pos_emb, neg_emb)
class_means_dict = get_class_means(pos_emb, neg_emb)

In [ ]:
layerwise_probe = get_probes(pos_emb, neg_emb)

In [ ]:
intervention_dict = get_interventions_dict(layerwise_probe, class_means_dict)

In [ ]:
print(len(intervention_dict))

In [ ]:
inference_fun = partial(vanila_inference, fschat=fschat, max_new_tokens=max_new_tokens)

In [ ]:
def get_answer_with_intervention(model, tokenizer, prompt, max_new_tokens=1024, interventions={}, intervention_fn=None):
    out = tokenizer(prompt, return_tensors="pt")
    input_ids = out.input_ids.cuda()
    attention_mask = out.attention_mask.cuda()
    # --- intervention code --- #
    def id(head_output, layer_name): 
        return head_output
    if interventions == {}: 
        intervene = id
        layers_to_intervene = []
    else: 
        intervene = partial(intervention_fn, interventions=interventions)
        layers_to_intervene = list(interventions.keys())
    # --- intervention code --- #
    input_token_len = input_ids.shape[1]
    with torch.inference_mode():
        with TraceDict(model, layers_to_intervene, edit_output=intervene) as ret: 
            model_output = model.generate(inputs = input_ids, 
                                          attention_mask = attention_mask,
                                          max_new_tokens=max_new_tokens,
                                          stopping_criteria=StoppingCriteriaList([StopOnTokens()]),
                                          use_cache=True,
                                          # repetition_penalty=1.1
                                         )
        outstr = tokenizer.decode(model_output[0], skip_special_tokens=True)
    torch.cuda.empty_cache()
    return outstr

In [ ]:
import json

for row_id, row in tqdm(df.iterrows()):
    print(f'########## {row_id} ##########')
    q = row['question']
    question_str = convert_to_quesion_str(q)
    print(question_str)
    question_str = fschat + "\n" + question_str
    try:
        with torch.inference_mode():
            vanila_output = inference_fun(raw_query=question_str, model=model, tokenizer=tokenizer)
        ans_query = vanila_output.split(question_str)[-1].strip().rstrip()
        print('VANILLA MODEL')
        print(ans_query)
        
        # out = get_answer_with_intervention(model, tokenizer, question_str, \
        #                                    max_new_tokens=max_new_tokens, interventions=intervention_dict, \
        #                                    intervention_fn=lt_modulated_proj).split(question_str)[-1].strip().rstrip()
        # print("OURS")
        # print(out)

        tmp = {'question': q,
               'vanilla': ans_query,
               # 'ours': out
              }
    

        if not os.path.exists(os.path.join(outdir)):
            os.makedirs(os.path.join(outdir))
        with open('{}/tqa_{}_res_{}.json'.format(outdir, 'mistral@7b', row_id), 'w') as f:
            f.write(json.dumps(tmp))
    except Exception as e:
        raise e

In [ ]:
print(a)